<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9b task runner — the 9a-matched hybrid/JANA SBIBM campaign

This is the shared execution engine used by the ten public
`Exercise_9b_SBIBM_<Task>.ipynb` notebooks. It is not the aggregate notebook:
`Exercise_9b_SBIBM.ipynb` only discovers completed tagged artifacts and builds
partial or complete comparisons.

The non-smoke campaign deliberately copies the controlled strategy from
Exercise 9a: **10,000 simulations**, four independently initialized flow
members per conditional density, ten independently initialized plain
three-class classifiers, batch-size row budget 32, 250 epochs, and a stepped
$10^{-4}\rightarrow10^{-9}$ learning rate. The flow topology is the original
Exercise-9 10-layer conditional RQS; the classifier is a uniform four-layer
1024-unit ReLU MLP. There is no dropout, weight decay, normalization layer,
residual block, auxiliary loss, normalization loss, or bridge loss.

The only SBIBM-specific preprocessing retained is fixed training-column
standardization for the classifier. This is needed because the ten tasks use
very different physical scales; it contains no analytic density, simulator
mean, score, or task-aware feature. Inference averages each member's positive
softmax probability quotient arithmetically, exactly as in Exercise 9a.

In [ ]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib
import importlib.util
import os, sys, subprocess
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("EX9B_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        SOURCE_ROOT = Path("/content")  # isolated per Colab runtime; safe in parallel
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9b_SBIBM"
        )
    else:
        SOURCE_ROOT = Path("/content")
        default_artifact_root = Path("/content/exercise_9b_SBIBM_artifacts")
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_DIR = SOURCE_ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        path = str(import_dir.resolve())
        if path not in sys.path:
            sys.path.insert(0, path)

    def installed_version(distribution):
        try:
            return package_version(distribution)
        except PackageNotFoundError:
            return None

    # Exercise 10 compatibility contract: resolve only the modern runtime
    # dependencies. sbibm 1.1.0's optional historical algorithm stack is
    # unused here and cannot be resolved on Python 3.12+.
    if installed_version("nflows") != "0.14" or importlib.util.find_spec("pyro") is None:
        run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14", "pyro-ppl")
    if installed_version("sbibm") != "1.1.0":
        run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "sbibm==1.1.0")

    # SIR and Lotka--Volterra import diffeqtorch at module import time.
    # Install only that small Python import layer. The SIR and
    # Lotka--Volterra launchers replace the legacy Julia solver with
    # audited Python compatibility backends before any simulation.
    if installed_version("julia") != "0.6.2" or importlib.util.find_spec("opt_einsum") is None:
        run(sys.executable, "-m", "pip", "install", "-q", "julia==0.6.2", "opt_einsum")
    if installed_version("diffeqtorch") != "1.0.0":
        run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "diffeqtorch==1.0.0")
    importlib.invalidate_caches()
    import nflows, pyro, sbibm
    assert installed_version("nflows") == "0.14"
    assert installed_version("sbibm") == "1.1.0"
    print(
        "Pinned runtime: nflows=0.14, sbibm=1.1.0 (--no-deps); "
        "legacy ODE task imports enabled"
    )
    os.chdir(TUTORIAL_DIR)
else:
    default_artifact_root = Path.cwd() / "exercise_9b_SBIBM_artifacts"
    for candidate in [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]:
        if (candidate / "utils_hnpe.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

ARTIFACT_ROOT = Path(
    os.environ.get("EX9B_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["EX9B_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
print("Working directory:", Path.cwd())
print("Persistent Exercise-9b artifact root:", ARTIFACT_ROOT)

In [ ]:
import copy
import gc
import hashlib
import inspect
import json
import math
import os
import random
import traceback
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sbibm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.spatial.distance import cdist, pdist
from torch.utils.data import DataLoader, TensorDataset

from utils_benchmark import (
    infer_parameter_transform,
    load_or_simulate_bank,
    split_simulation_bank,
    task_recommendation_table,
)
from utils_hnpe import (
    sample_spline_flow_ensemble,
    spline_flow_ensemble_log_prob,
    train_spline_flow_ensemble,
)
from utils_plotting import export_standalone_figure_script

from utils_exercise9b_contract import DEFAULT_SEED, MAX_BASE_SEED
REQUESTED_TASK = os.environ.get("EX9B_TASK", "").lower()
SIMULATOR_BACKEND_DIAGNOSTICS = None
if (
    REQUESTED_TASK == "sir"
    and os.environ.get("EX9B_SIR_BACKEND", "").lower() == "python_rk4"
):
    from utils_sir_backend import install_sir_python_backend
    SIMULATOR_BACKEND_DIAGNOSTICS = install_sir_python_backend(sbibm)
elif (
    REQUESTED_TASK == "lotka_volterra"
    and os.environ.get("EX9B_LOTKA_VOLTERRA_BACKEND", "").lower()
    == "python_logrk4"
):
    from utils_lotka_volterra_backend import (
        install_lotka_volterra_python_backend,
    )
    SIMULATOR_BACKEND_DIAGNOSTICS = (
        install_lotka_volterra_python_backend(sbibm)
    )
SIMULATOR_BACKEND = (
    os.environ.get("EX9B_SIMULATOR_BACKEND", "sbibm_default")
    if REQUESTED_TASK in {"sir", "lotka_volterra"}
    else "sbibm_default"
)
SEED = int(os.environ.get("EX9B_SEED", str(DEFAULT_SEED)))
if not 0 <= SEED <= MAX_BASE_SEED:
    raise ValueError(
        f"EX9B_SEED must lie in [0, {MAX_BASE_SEED}] so every derived "
        "legacy NumPy/sklearn seed remains valid."
    )
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % 2**32)
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

seed_everything(SEED)
print("sbibm version:", sbibm.__version__)
print("Using device:", device)
print("Simulator backend:", SIMULATOR_BACKEND)
if SIMULATOR_BACKEND_DIAGNOSTICS is not None:
    print(
        "Simulator backend preflight:",
        json.dumps(SIMULATOR_BACKEND_DIAGNOSTICS, indent=2),
    )

## 1. A controlled pure-JANA versus hybrid-CE experiment

In latent parameter coordinates $z=T(\theta)$, the learned reference models are normalized conditional flows $q_\phi(z\mid x)$ and $q_\eta(x\mid z)$. Both conditionals use the **entire vector** on each side. The matched-capacity pure-JANA route is

$$z\sim q_\phi(z\mid x),\qquad x_{\rm rep}\sim q_\eta(x\mid z).$$

For the benchmark experiments JANA sets the summary-space MMD coefficient to zero. Its density objective is the sum of posterior and likelihood NLLs. Because the two flows here have disjoint parameters, fitting the two NLLs separately is algebraically equivalent to optimizing that sum. Ensembling, parameter transforms, training rows, and checkpoint selection are held fixed for both routes; this deliberately compares direct JANA inference with CE correction rather than software frameworks.

The hybrid route additionally forms the three equal-prior class laws

$$
S=\rho_z(z)p(x\mid z),\qquad
P=m(x)q_\phi(z\mid x),\qquad
L=\rho_z(z)q_\eta(x\mid z).
$$

Each of the ten independently initialized ordinary MLPs returns three logits and is trained only with

$$\mathcal L_{\rm CE}=-\mathbb E\log D_Y(z,x),\qquad
(D_S,D_P,D_L)=\operatorname{softmax}f_\psi(z,x).$$

Equal class priors then give $S/P=D_S/D_P$ and $S/L=D_S/D_L$. Every MLP is a uniform stack of `Linear` and ReLU layers followed by three outputs. It has no dropout, weight decay, LayerNorm or BatchNorm, skip connection, special output initialization, or output clipping. A fixed training-column mean/standard-deviation transform is retained only because the SBIBM tasks use very different coordinate scales; it is a change of input units, not a trainable normalization layer or density-aware feature.

For a strict paired comparison, each hybrid posterior reweights the same finite $q_\phi$ proposal bank from which the direct-JANA sample is drawn. The methods also share the same observation jitters, frozen four-member flow mixtures, and official evaluation reference. The deployed correction is the arithmetic mean of the ten member-wise positive softmax quotients.


## 2. Post-training normalization and bridge checks

The two reference flows are normalized by construction. Pure JANA uses them without any correction. With the equal-prior softmax classifier, the hybrid ratios should additionally obey

$$Z_P(x)=\mathbb E_{q_\phi(z\mid x)}\frac{D_S}{D_P}=1,\qquad
Z_L(z)=\mathbb E_{q_\eta(x\mid z)}\frac{D_S}{D_L}=1.$$

These identities are **not losses**. After every CE checkpoint is fixed, independent Monte Carlo banks anchored on held-out grouped rows provide two estimates of each $Z$. We report the cross estimate $(Z_A-1)(Z_B-1)$ and a pooled positive square for plotting. Neither statistic can affect gradients, early stopping, or model selection.

The raw pure-JANA evidence path is

$$
\log m_{\rm JANA}(x;z)=
\log\rho_z(z)+\log q_\eta(x\mid z)-\log q_\phi(z\mid x).
$$

The hybrid path adds

$$\log\frac{D_S}{D_L}-\log\frac{D_S}{D_P}=\log D_P-\log D_L.$$

Both paths should be independent of $z$ at fixed $x$. We therefore compare their held-out variances on exactly the same IID audit bank. This is a consistency check, not a training target or prediction weight. At inference time, finite-bank hybrid weights are obtained directly from $D_S/D_P$ or $D_S/D_L$ and self-normalized; no exponential of a learned log ratio is evaluated.


In [ ]:
from utils_exercise9b_contract import (
    ALL_TASKS,
    CAMPAIGN_SCHEMA,
    INITIAL_LEARNING_RATE,
    JANA_PAPER_COMMIT,
    LEARNING_RATE_DROP_FACTOR,
    LEARNING_RATE_STEP_EPOCHS,
    METHOD_HYBRID,
    METHOD_JANA,
    METHOD_LABELS,
    METHODS,
    METRIC_SCHEMA,
    MINIMUM_LEARNING_RATE,
    PROFILES,
    campaign_run_tag,
    campaign_signature,
    expected_status,
    normalize_profile,
)

PROFILE = normalize_profile(os.environ.get("EX9B_PROFILE", "PAPER"))
TASK_NAME = os.environ.get("EX9B_TASK", "two_moons")
LOAD_IF_AVAILABLE = os.environ.get("EX9B_LOAD_IF_AVAILABLE", "0") == "1"
FAIL_ON_TASK_ERROR = os.environ.get("EX9B_FAIL_ON_TASK_ERROR", "1") != "0"
if TASK_NAME not in ALL_TASKS:
    raise ValueError(f"TASK_NAME must be one of {ALL_TASKS}")

campaign = PROFILES[PROFILE]
TASKS_TO_RUN = (TASK_NAME,)
CAMPAIGN_SIGNATURE = campaign_signature(PROFILE)
RUN_TAG = campaign_run_tag(PROFILE, SEED)

if PROFILE != "SMOKE":
    if campaign["num_simulations"] != 10_000:
        raise ValueError("The 9a-matched campaign must retain exactly 10,000 simulations.")
    if campaign["flow_members"] != 4 or campaign["classifier_members"] != 10:
        raise ValueError("The 9a-matched campaign requires four flows and ten classifiers.")
    if campaign["flow_epochs"] != 250 or campaign["class_epochs"] != 250:
        raise ValueError("The 9a-matched campaign requires 250 flow and classifier epochs.")
    if campaign["flow_batch_size"] != 32 or campaign["classifier_batch_size"] != 32:
        raise ValueError("The 9a-matched campaign requires batch-size row budget 32.")

ARTIFACT_ROOT = Path(os.environ["EX9B_ARTIFACT_ROOT"]).expanduser().resolve()
MODEL_ROOT = ARTIFACT_ROOT / "models"
CACHE_ROOT = ARTIFACT_ROOT / "simulation_banks"
RESULT_ROOT = ARTIFACT_ROOT / "results"
FIGURE_ROOT = ARTIFACT_ROOT / "figures_scripts"
for directory in (MODEL_ROOT, CACHE_ROOT, RESULT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


def scheduled_learning_rate(epoch):
    if int(epoch) < 0:
        raise ValueError("Epoch must be non-negative.")
    return max(
        MINIMUM_LEARNING_RATE,
        INITIAL_LEARNING_RATE
        * LEARNING_RATE_DROP_FACTOR ** (
            int(epoch) // LEARNING_RATE_STEP_EPOCHS
        ),
    )


def set_optimizer_learning_rate(optimizer, epoch):
    value = scheduled_learning_rate(epoch)
    for group in optimizer.param_groups:
        group["lr"] = value
    return value


recommendations = task_recommendation_table().copy()
lv_row = recommendations["task"] == "lotka_volterra"
recommendations.loc[lv_row, "data_type"] = "continuous LogNormal-noised trajectories"
recommendations.loc[lv_row, "reason"] = (
    "The official simulator returns continuous LogNormal-noised trajectories; "
    "no dequantization is applied."
)
if SIMULATOR_BACKEND.startswith("sbibm_lotka_volterra_python_logrk4"):
    recommendations.loc[lv_row, "colab_status"] = "audited Python backend"
    recommendations.loc[lv_row, "reason"] = (
        "The official Lotka--Volterra model and LogNormal observation "
        "law are retained; only the incompatible Julia solver is replaced."
    )
sir_row = recommendations["task"] == "sir"
if SIMULATOR_BACKEND.startswith("sbibm_sir_python_rk4"):
    recommendations.loc[sir_row, "colab_status"] = "audited Python backend"
    recommendations.loc[sir_row, "reason"] = (
        "The official SIR model and Binomial observation law are retained; "
        "only the incompatible Julia ODE solver is replaced."
    )
display(recommendations.loc[recommendations["task"] == TASK_NAME].style.hide(axis="index"))
print(json.dumps({
    "profile": PROFILE,
    "task": TASK_NAME,
    "seed": SEED,
    "run_tag": RUN_TAG,
    "campaign_signature": CAMPAIGN_SIGNATURE,
    "simulator_backend": SIMULATOR_BACKEND,
    "artifact_root": str(ARTIFACT_ROOT),
    "models": str(MODEL_ROOT),
    "results": str(RESULT_ROOT),
    "simulations": campaign["num_simulations"],
    "flow_members": campaign["flow_members"],
    "classifier_members": campaign["classifier_members"],
    "flow_epochs": campaign["flow_epochs"],
    "classifier_epochs": campaign["class_epochs"],
    "flow_batch_size": campaign["flow_batch_size"],
    "classifier_row_batch_budget": campaign["classifier_batch_size"],
    "learning_rate": "1e-4 / 10 every 40 epochs, floor 1e-9",
    "hybrid_loss": "equal-prior multiclass CE only",
    "ratio_ensemble": "arithmetic mean of member-wise softmax quotients",
    "normalization": "post-training correction/check only",
    "bridge": "post-training consistency check only",
}, indent=2))

## 3. Discrete observations: an explicit change of measure

`bernoulli_glm_raw` and SIR live on an integer lattice.  We use randomized dequantization $\widetilde x=x+u$ with $u_j\sim\mathrm{Uniform}[-w_j/2,w_j/2]$ and train a **continuous density of $\widetilde x$**.  For raw binary/count data $w_j=1$, so integrating a perfect dequantized density over the unit cell recovers a probability mass.  At an observed integer vector, posterior evaluation averages over multiple cell jitters rather than pretending the cell center is a continuous observation.

`bernoulli_glm` is a derived sufficient statistic on an irregular finite support.  Its data-driven rectangular jitter is only a declared smoothing convention; it is **not** an exact PMF representation.  Accordingly, density/evidence values from that task must not be labeled exact likelihoods.  C2ST and MMD against the official posterior remain meaningful tests of the resulting inference procedure.

Official `sbibm` Lotka--Volterra observations are continuous LogNormal-noised trajectories.  They therefore receive zero dequantization width and no observation jitter in training, posterior evaluation, or predictive evaluation.  The bootstrap installs `sbibm==1.1.0` with `--no-deps`, following Exercise 10, plus only the lightweight Python import layer for `diffeqtorch`.  The SIR and Lotka--Volterra launchers select audited Python solvers for their unchanged official equations and observation models; official observations and reference-posterior samples remain untouched.  The selected backend and its numerical preflight are recorded in the status manifest.  A failure writes that JSON manifest and, by default, raises after displaying it.


In [ ]:
DISCRETE_TASKS = {"bernoulli_glm", "bernoulli_glm_raw", "sir"}

def require_finite_rows(task_name, stage, values):
    if torch.is_tensor(values):
        values = values.detach().cpu().numpy()
    array = np.asarray(values)
    if array.ndim == 0:
        array = array.reshape(1, 1)
    elif array.ndim == 1:
        array = array.reshape(-1, 1)
    flat = array.reshape(len(array), -1)
    finite_rows = np.isfinite(flat).all(axis=1)
    if not finite_rows.all():
        bad = int((~finite_rows).sum())
        raise FloatingPointError(
            f"{task_name} [{stage}]: {bad}/{len(flat)} rows contain non-finite values"
        )
    return array

def dequantization_widths(task_name, values):
    values = np.asarray(values, dtype=np.float32)
    if task_name not in DISCRETE_TASKS:
        return np.zeros(values.shape[1], dtype=np.float32)
    if task_name != "bernoulli_glm":
        return np.ones(values.shape[1], dtype=np.float32)
    # The sufficient-statistic support is irregular.  This deterministic
    # local scale defines smoothing only; it does not claim PMF semantics.
    widths = []
    for column in values.T:
        unique = np.unique(np.round(column[: min(len(column), 20_000)], 7))
        gaps = np.diff(unique)
        gaps = gaps[gaps > 1.0e-7]
        robust_scale = np.std(column, dtype=np.float64)
        width = np.median(gaps) if len(gaps) else 0.02 * robust_scale
        width = np.clip(width, 1.0e-4, max(1.0e-4, 0.10 * robust_scale))
        widths.append(width)
    return np.asarray(widths, dtype=np.float32)

def dequantize(values, widths, seed):
    values = np.asarray(values, dtype=np.float32)
    widths = np.asarray(widths, dtype=np.float32)
    if not np.any(widths):
        return values.copy()
    rng = np.random.default_rng(int(seed))
    return (values + (rng.random(values.shape) - 0.5) * widths).astype(np.float32)

def install_inverse_rqs_float64_retry():
    """Retry the same finite inverse-RQS tensors in float64 on cancellation."""
    import functools
    import importlib
    module = importlib.import_module("nflows.transforms.splines.rational_quadratic")
    original = module.rational_quadratic_spline
    if getattr(original, "_ex9b_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get("inverse", False)
            if not inverse or inputs.dtype != torch.float32:
                raise
            floating = [v for v in (*args, *kwargs.values()) if torch.is_tensor(v) and v.is_floating_point()]
            if any(not bool(torch.isfinite(v).all()) for v in floating):
                raise FloatingPointError("Non-finite tensor reached inverse RQS") from error32
            convert = lambda v: v.double() if torch.is_tensor(v) and v.is_floating_point() else v
            try:
                output, logdet = original(
                    *(convert(v) for v in args),
                    **{k: convert(v) for k, v in kwargs.items()},
                )
            except AssertionError as error64:
                raise RuntimeError("Inverse RQS discriminant also failed in float64") from error64
            if not bool(torch.isfinite(output).all() and torch.isfinite(logdet).all()):
                raise FloatingPointError("Float64 inverse RQS returned non-finite values")
            guarded.retry_count += 1
            if guarded.retry_count == 1:
                warnings.warn("Inverse-RQS cancellation: retrying identical tensors in float64.", RuntimeWarning)
            return output.to(inputs.dtype), logdet.to(inputs.dtype)

    guarded._ex9b_float64_retry = True
    guarded.retry_count = 0
    module.rational_quadratic_spline = guarded
    importlib.import_module("nflows.transforms.splines").rational_quadratic_spline = guarded
    importlib.import_module("nflows.transforms.autoregressive").rational_quadratic_spline = guarded
    return guarded

RQS_GUARD = install_inverse_rqs_float64_retry()


In [ ]:
CLASS_S, CLASS_P, CLASS_L = 0, 1, 2

class PlainThreeClassMLP(nn.Module):
    """One ordinary MLP with three unconstrained class logits."""
    def __init__(self, input_dim, width, hidden_layers):
        super().__init__()
        modules = []
        in_features = int(input_dim)
        for _ in range(int(hidden_layers)):
            modules.extend([nn.Linear(in_features, int(width)), nn.ReLU()])
            in_features = int(width)
        modules.append(nn.Linear(in_features, 3))
        self.network = nn.Sequential(*modules)

    def forward(self, values):
        return self.network(values)

def model_parameter_counts(model):
    return {"total": sum(parameter.numel() for parameter in model.parameters())}

preview_model = PlainThreeClassMLP(
    12, campaign["class_width"], campaign["class_layers"]
)
print("Plain three-class MLP parameter count:", model_parameter_counts(preview_model))
del preview_model


In [ ]:
def flow_model_config(n_target):
    # Exactly the simple RQS family used by the original dual hNPE--hNDE
    # Exercise 9. For a scalar target, utils_hnpe uses the one-dimensional
    # autoregressive RQS analogue because a coupling mask needs >=2 features.
    return {
        "n_coupling_layers": 10,
        "hidden_features": 512,
        "hidden_layers": 4,
        "spline_num_bins": 16,
        "spline_tail_bound": 5.0,
        "dropout_probability": 0.0,
    }

FLOW_TRAINING_CONFIG = {
    "batch_size": campaign["flow_batch_size"],
    "n_epochs": campaign["flow_epochs"],
    "learning_rate": INITIAL_LEARNING_RATE,
    "min_learning_rate": MINIMUM_LEARNING_RATE,
    "lr_scheduler": "step",
    "lr_scheduler_factor": LEARNING_RATE_DROP_FACTOR,
    "lr_scheduler_patience": LEARNING_RATE_STEP_EPOCHS,
    "validation_fraction": 0.2,
    "patience": campaign["training_patience"],
    "print_every": LEARNING_RATE_STEP_EPOCHS,
    "gradient_clip": 5.0,
    "weight_decay": 0.0,
}

def draw_flow_mixture(flow_packs, n_samples, contexts, seed, allocation="balanced"):
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    seed_everything(seed)
    values = sample_spline_flow_ensemble(
        flow_packs, int(n_samples), context=contexts, seed=int(seed),
        batch_size=16_384, allocation=str(allocation),
    )
    values = np.asarray(values, dtype=np.float32)
    n_features = int(np.asarray(flow_packs[0]["target_scaler"].mean).size)
    if len(contexts) == 1:
        values = values.reshape(1, int(n_samples), n_features)
    expected = (len(contexts), int(n_samples), n_features)
    if values.shape != expected:
        raise RuntimeError(f"Unexpected mixture sample shape {values.shape}; expected {expected}")
    if not np.isfinite(values).all():
        raise FloatingPointError("Flow-mixture sampling returned non-finite values")
    return values

def prior_log_prob_latent(task, transform, latent):
    latent = np.asarray(latent, dtype=np.float32)
    theta = transform.inverse(latent)
    with torch.no_grad():
        values = task.get_prior_dist().log_prob(torch.as_tensor(theta, dtype=torch.float32))
    values = values.detach().cpu().numpy()
    if values.ndim > 1:
        values = values.sum(axis=tuple(range(1, values.ndim)))
    return np.asarray(values, dtype=np.float64) + transform.log_abs_det_inverse(latent)

def build_class_groups(z, x, q_phi, q_eta, seed):
    z = np.asarray(z, dtype=np.float32)
    x = np.asarray(x, dtype=np.float32)
    z_p = draw_flow_mixture(q_phi, 1, x, seed + 1)[:, 0, :]
    x_l = draw_flow_mixture(q_eta, 1, z, seed + 2)[:, 0, :]
    simulator = np.column_stack([z, x])
    posterior_reference = np.column_stack([z_p, x])
    likelihood_reference = np.column_stack([z, x_l])
    groups = np.stack([simulator, posterior_reference, likelihood_reference], axis=1)
    if not np.isfinite(groups).all():
        raise FloatingPointError("Non-finite class group")
    return groups.astype(np.float32)

In [ ]:
def _anchor_rows(values, n_rows, rng):
    replace = int(n_rows) > len(values)
    return values[rng.choice(len(values), size=int(n_rows), replace=replace)]

def build_post_training_diagnostic_bundle(task, transform, z, x, q_phi, q_eta, seed):
    input_dim = z.shape[1] + x.shape[1]
    memory_factor = max(1, int(math.ceil(input_dim / 24)))
    norm_groups = max(24, campaign["norm_groups"] // memory_factor)
    norm_inner = max(4, int(campaign["norm_inner"] / math.sqrt(memory_factor)))
    bridge_groups = max(24, campaign["bridge_groups"] // memory_factor)
    bridge_inner = max(4, int(campaign["bridge_inner"] / math.sqrt(memory_factor)))
    rng = np.random.default_rng(int(seed))

    x_anchor = _anchor_rows(x, norm_groups, rng)
    zp_a = draw_flow_mixture(q_phi, norm_inner, x_anchor, seed + 1)
    zp_b = draw_flow_mixture(q_phi, norm_inner, x_anchor, seed + 2)
    x_repeat = np.repeat(x_anchor[:, None, :], norm_inner, axis=1)

    z_anchor = _anchor_rows(z, norm_groups, rng)
    zl_a = draw_flow_mixture(q_eta, norm_inner, z_anchor, seed + 3)
    zl_b = draw_flow_mixture(q_eta, norm_inner, z_anchor, seed + 4)
    z_repeat = np.repeat(z_anchor[:, None, :], norm_inner, axis=1)

    x_bridge = _anchor_rows(x, bridge_groups, rng)
    z_bridge = draw_flow_mixture(
        q_phi, bridge_inner, x_bridge, seed + 5, allocation="iid"
    )
    xb_repeat = np.repeat(x_bridge[:, None, :], bridge_inner, axis=1)
    flat_z = z_bridge.reshape(-1, z.shape[1])
    flat_x = xb_repeat.reshape(-1, x.shape[1])
    bridge_base = (
        prior_log_prob_latent(task, transform, flat_z)
        + spline_flow_ensemble_log_prob(q_eta, flat_x, context=flat_z)
        - spline_flow_ensemble_log_prob(q_phi, flat_z, context=flat_x)
    ).reshape(bridge_groups, bridge_inner)

    bundle = {
        "zp_a": np.concatenate([zp_a, x_repeat], axis=2),
        "zp_b": np.concatenate([zp_b, x_repeat], axis=2),
        "zl_a": np.concatenate([z_repeat, zl_a], axis=2),
        "zl_b": np.concatenate([z_repeat, zl_b], axis=2),
        "bridge": np.concatenate([z_bridge, xb_repeat], axis=2),
        "bridge_base": bridge_base.astype(np.float32),
    }
    for name, values in bundle.items():
        if not np.isfinite(values).all():
            raise FloatingPointError(f"Non-finite post-training diagnostic array {name}")
    return bundle

def fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = points.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale

def transform_classifier_points(points, center, scale):
    points = np.asarray(points, dtype=np.float32)
    return ((points - center) / scale).astype(np.float32)

def prepare_diagnostic_bundle(bundle, center, scale):
    prepared = {}
    for name, values in bundle.items():
        values = np.asarray(values, dtype=np.float32)
        if name != "bridge_base":
            values = transform_classifier_points(values, center, scale)
        prepared[name] = torch.as_tensor(values, dtype=torch.float32)
    return prepared

def _probability_odds_torch(probabilities, numerator, denominator):
    numerator_values = probabilities[..., int(numerator)]
    denominator_values = probabilities[..., int(denominator)]
    tiny = torch.finfo(probabilities.dtype).tiny
    odds = numerator_values / denominator_values.clamp_min(tiny)
    if not bool(torch.isfinite(odds).all()):
        raise FloatingPointError("Non-finite softmax probability quotient")
    return odds

@torch.no_grad()
def constraint_diagnostic_terms(model, bundle, index):
    def outputs(name):
        points = bundle[name][index].to(device)
        shape = points.shape[:-1]
        logits = model(points.reshape(-1, points.shape[-1])).double()
        probabilities = torch.softmax(logits, dim=1).reshape(*shape, 3)
        log_probabilities = torch.log_softmax(logits, dim=1).reshape(*shape, 3)
        return probabilities, log_probabilities

    probabilities_zp_a, _ = outputs("zp_a")
    probabilities_zp_b, _ = outputs("zp_b")
    probabilities_zl_a, _ = outputs("zl_a")
    probabilities_zl_b, _ = outputs("zl_b")
    odds_zp_a = _probability_odds_torch(probabilities_zp_a, CLASS_S, CLASS_P)
    odds_zp_b = _probability_odds_torch(probabilities_zp_b, CLASS_S, CLASS_P)
    odds_zl_a = _probability_odds_torch(probabilities_zl_a, CLASS_S, CLASS_L)
    odds_zl_b = _probability_odds_torch(probabilities_zl_b, CLASS_S, CLASS_L)
    dzp_a, dzp_b = odds_zp_a.mean(dim=1) - 1.0, odds_zp_b.mean(dim=1) - 1.0
    dzl_a, dzl_b = odds_zl_a.mean(dim=1) - 1.0, odds_zl_b.mean(dim=1) - 1.0
    cross = 0.5 * ((dzp_a * dzp_b).mean() + (dzl_a * dzl_b).mean())
    monitor = 0.5 * (
        (0.5 * (dzp_a + dzp_b)).square().mean()
        + (0.5 * (dzl_a + dzl_b)).square().mean()
    )
    _, log_probabilities_bridge = outputs("bridge")
    # c-a = log(D_S/D_L)-log(D_S/D_P) = log D_P-log D_L.
    log_correction = (
        log_probabilities_bridge[..., CLASS_P]
        - log_probabilities_bridge[..., CLASS_L]
    )
    raw_evidence = bundle["bridge_base"][index].to(device).double() + log_correction
    bridge = raw_evidence.var(dim=1, unbiased=True).mean()
    if not bool(torch.isfinite(cross) and torch.isfinite(monitor) and torch.isfinite(bridge)):
        raise FloatingPointError("Non-finite post-training consistency diagnostic")
    return cross.float(), monitor.float(), bridge.float()


In [ ]:
def classifier_fingerprint(train_groups, validation_groups, config):
    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode("utf-8"))
    for values in [train_groups, validation_groups]:
        array = np.ascontiguousarray(values)
        digest.update(str(array.shape).encode("ascii"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(array.view(np.uint8))
    return digest.hexdigest()

def safe_torch_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def class_ce(model, groups):
    groups = groups.to(device)
    flat = groups.reshape(-1, groups.shape[-1])
    logits = model(flat).reshape(len(groups), 3, 3)
    labels = torch.arange(3, device=device).repeat(len(groups))
    return F.cross_entropy(logits.reshape(-1, 3), labels)

def validation_classification_objective(model, groups, chunk_size=512):
    model.eval()
    ce_sum = 0.0
    with torch.no_grad():
        for start in range(0, len(groups), chunk_size):
            batch = groups[start:start + chunk_size]
            ce = class_ce(model, batch)
            ce_sum += float(ce.cpu()) * len(batch)
    return ce_sum / len(groups)

def member_constraint_diagnostics(model, bundle, chunk_size=128):
    # This helper is intentionally separate from the CE objective and is
    # called only after the CE-selected checkpoint is frozen.
    model.eval()
    cross_sum = monitor_sum = bridge_sum = count = 0.0
    n_eval = max(len(bundle["zp_a"]), len(bundle["bridge"]))
    for start in range(0, n_eval, int(chunk_size)):
        stop = min(n_eval, start + int(chunk_size))
        raw = torch.arange(start, stop)
        index_norm = raw % len(bundle["zp_a"])
        index_bridge = raw % len(bundle["bridge"])
        compact = {
            "zp_a": bundle["zp_a"][index_norm],
            "zp_b": bundle["zp_b"][index_norm],
            "zl_a": bundle["zl_a"][index_norm],
            "zl_b": bundle["zl_b"][index_norm],
            "bridge": bundle["bridge"][index_bridge],
            "bridge_base": bundle["bridge_base"][index_bridge],
        }
        local = torch.arange(stop - start)
        cross, monitor, bridge = constraint_diagnostic_terms(model, compact, local)
        weight = stop - start
        cross_sum += float(cross.cpu()) * weight
        monitor_sum += float(monitor.cpu()) * weight
        bridge_sum += float(bridge.cpu()) * weight
        count += weight
    return {
        "norm_cross": cross_sum / count,
        "norm_monitor": monitor_sum / count,
        "bridge": bridge_sum / count,
    }

def train_plain_classifier(
    train_groups, validation_groups, checkpoint_dir, seed,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    flat = train_groups.reshape(-1, train_groups.shape[-1])
    center, scale = fit_classifier_transform(flat)
    train_tensor = torch.as_tensor(
        transform_classifier_points(train_groups, center, scale), dtype=torch.float32
    )
    validation_tensor = torch.as_tensor(
        transform_classifier_points(validation_groups, center, scale), dtype=torch.float32
    )
    config = {
        "input_dim": int(train_groups.shape[-1]),
        "width": int(campaign["class_width"]),
        "hidden_layers": int(campaign["class_layers"]),
        "epochs": int(campaign["class_epochs"]),
        "initial_learning_rate": INITIAL_LEARNING_RATE,
        "minimum_learning_rate": MINIMUM_LEARNING_RATE,
        "learning_rate_drop_factor": LEARNING_RATE_DROP_FACTOR,
        "learning_rate_step_epochs": LEARNING_RATE_STEP_EPOCHS,
        "batch_size_row_budget": int(campaign["classifier_batch_size"]),
        "patience": int(campaign["training_patience"]),
        "objective": "equal_prior_multiclass_ce_only",
        "campaign_signature": CAMPAIGN_SIGNATURE,
        "campaign_seed": SEED,
        "normalization_role": "post_training_only",
        "bridge_role": "post_training_check_only",
        "input_transform": "fixed_column_mean_std_v1",
        "regularization": "none",
    }
    members = []
    for member in range(campaign["classifier_members"]):
        member_seed = int(seed + 10_007 * member)
        member_config = {
            **config, "member_index": int(member), "member_seed": member_seed,
        }
        fingerprint = classifier_fingerprint(
            train_groups, validation_groups, member_config
        )
        checkpoint = checkpoint_dir / f"member_{member:02d}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            saved = safe_torch_load(checkpoint)
            if saved.get("fingerprint") != fingerprint:
                raise RuntimeError(f"Classifier checkpoint fingerprint mismatch: {checkpoint}")
            if (
                saved.get("member_index") != member
                or saved.get("member_seed") != member_seed
                or saved.get("config") != member_config
            ):
                raise RuntimeError(f"Classifier member metadata mismatch: {checkpoint}")
            model = PlainThreeClassMLP(**saved["config_model"]).to(device)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            if saved.get("input_transform") != "fixed_column_mean_std_v1":
                raise RuntimeError(f"Classifier checkpoint uses a stale input transform: {checkpoint}")
            saved_center = np.asarray(saved.get("center"), dtype=np.float32)
            saved_scale = np.asarray(saved.get("scale"), dtype=np.float32)
            if (
                saved_center.shape != center.shape
                or saved_scale.shape != scale.shape
                or not np.isfinite(saved_center).all()
                or not np.isfinite(saved_scale).all()
                or not np.all(saved_scale > 0.0)
                or not np.array_equal(saved_center, center)
                or not np.array_equal(saved_scale, scale)
            ):
                raise RuntimeError(f"Classifier input-transform mismatch: {checkpoint}")
            members.append({
                "model": model, "center": saved_center, "scale": saved_scale,
                "history": saved["history"],
            })
            print("Loaded", checkpoint)
            continue

        seed_everything(member_seed)
        model = PlainThreeClassMLP(
            config["input_dim"], config["width"], config["hidden_layers"]
        ).to(device)
        # Plain Adam: no weight decay and no auxiliary objective.
        optimizer = torch.optim.Adam(
            model.parameters(), lr=INITIAL_LEARNING_RATE
        )
        generator = torch.Generator().manual_seed(member_seed + 1)
        loader = DataLoader(
            TensorDataset(train_tensor),
            batch_size=max(1, int(campaign["classifier_batch_size"]) // 3),
            shuffle=True, generator=generator,
        )
        history = {"train_ce": [], "validation_ce": [], "learning_rate": []}
        best_state, best_value, stale = None, math.inf, 0
        patience = int(campaign["training_patience"])
        for epoch in range(int(campaign["class_epochs"])):
            learning_rate = set_optimizer_learning_rate(optimizer, epoch)
            running = 0.0
            rows = 0
            model.train()
            for (group_batch,) in loader:
                optimizer.zero_grad(set_to_none=True)
                ce = class_ce(model, group_batch)
                if not bool(torch.isfinite(ce)):
                    raise FloatingPointError("Non-finite multiclass CE")
                ce.backward()
                optimizer.step()
                running += float(ce.detach().cpu()) * len(group_batch)
                rows += len(group_batch)
            validation_ce = validation_classification_objective(model, validation_tensor)
            history["train_ce"].append(running / rows)
            history["validation_ce"].append(validation_ce)
            history["learning_rate"].append(optimizer.param_groups[0]["lr"])
            if validation_ce < best_value - 1.0e-6:
                best_value = validation_ce
                best_state = copy.deepcopy(model.state_dict())
                stale = 0
            else:
                stale += 1
            if (
                epoch == 0
                or (epoch + 1) % LEARNING_RATE_STEP_EPOCHS == 0
                or epoch + 1 == int(campaign["class_epochs"])
            ):
                print(
                    f"classifier {member + 1}/{campaign['classifier_members']} "
                    f"epoch {epoch + 1:03d}: CE={history['train_ce'][-1]:.4f}, "
                    f"val_CE={validation_ce:.4f}, lr={learning_rate:.1e}"
                )
            if stale >= patience:
                break
        if best_state is None:
            raise RuntimeError("Classifier training never produced a finite checkpoint")
        model.load_state_dict(best_state)
        model.eval()
        torch.save({
            "state_dict": best_state,
            "config_model": {
                "input_dim": config["input_dim"], "width": config["width"],
                "hidden_layers": config["hidden_layers"],
            },
            "config": member_config, "center": center, "scale": scale,
            "input_transform": "fixed_column_mean_std_v1",
            "history": history, "fingerprint": fingerprint,
            "member_index": member, "member_seed": member_seed,
        }, checkpoint)
        members.append({
            "model": model, "center": center, "scale": scale,
            "history": history,
        })
    return members

In [ ]:
def predict_class_probability_ratios(classifier, points, batch_size=16_384):
    """Arithmetic mean of member-wise direct float64 softmax quotients."""
    if not classifier:
        raise RuntimeError("At least one ratio classifier is required")
    points = np.asarray(points, dtype=np.float32)
    posterior_chunks, likelihood_chunks = [], []
    tiny = torch.finfo(torch.float64).tiny
    for start in range(0, len(points), int(batch_size)):
        stop = start + int(batch_size)
        posterior_mean = None
        likelihood_mean = None
        for pack in classifier:
            pack["model"].eval()
            standardized = transform_classifier_points(
                points[start:stop], pack["center"], pack["scale"]
            )
            with torch.no_grad():
                tensor = torch.as_tensor(standardized, device=device)
                probability = torch.softmax(
                    pack["model"](tensor).to(torch.float64), dim=1
                )
                posterior = probability[:, CLASS_S] / probability[:, CLASS_P].clamp_min(tiny)
                likelihood = probability[:, CLASS_S] / probability[:, CLASS_L].clamp_min(tiny)
            scale = float(len(classifier))
            posterior_mean = (
                posterior / scale if posterior_mean is None
                else posterior_mean + posterior / scale
            )
            likelihood_mean = (
                likelihood / scale if likelihood_mean is None
                else likelihood_mean + likelihood / scale
            )
        posterior_chunks.append(posterior_mean.detach().cpu().numpy())
        likelihood_chunks.append(likelihood_mean.detach().cpu().numpy())
    ratios = np.column_stack([
        np.concatenate(posterior_chunks), np.concatenate(likelihood_chunks)
    ])
    if not np.isfinite(ratios).all() or not np.all(ratios > 0.0):
        raise FloatingPointError("Invalid arithmetic softmax-ratio ensemble")
    return ratios


def predict_class_log_ratios(classifier, points, batch_size=16_384):
    """Log of the arithmetic ratio ensemble for log-density identities only."""
    ratios = predict_class_probability_ratios(
        classifier, points, batch_size=batch_size
    )
    values = np.log(np.maximum(ratios, np.finfo(np.float64).tiny))
    if not np.isfinite(values).all():
        raise FloatingPointError("Non-finite log arithmetic ratio ensemble")
    return values


def deployed_classifier_constraint_diagnostics(classifier, bundle):
    """Post-training checks of the deployed arithmetic ratio ensemble."""
    def ratios(name):
        values = np.asarray(bundle[name], dtype=np.float32)
        shape = values.shape[:-1]
        return predict_class_probability_ratios(
            classifier, values.reshape(-1, values.shape[-1])
        ).reshape(*shape, 2)

    odds_zp_a = ratios("zp_a")[..., 0]
    odds_zp_b = ratios("zp_b")[..., 0]
    odds_zl_a = ratios("zl_a")[..., 1]
    odds_zl_b = ratios("zl_b")[..., 1]
    dzp_a, dzp_b = odds_zp_a.mean(axis=1) - 1.0, odds_zp_b.mean(axis=1) - 1.0
    dzl_a, dzl_b = odds_zl_a.mean(axis=1) - 1.0, odds_zl_b.mean(axis=1) - 1.0
    cross = 0.5 * (np.mean(dzp_a * dzp_b) + np.mean(dzl_a * dzl_b))
    monitor = 0.5 * (
        np.mean(np.square(0.5 * (dzp_a + dzp_b)))
        + np.mean(np.square(0.5 * (dzl_a + dzl_b)))
    )
    bridge_values = np.asarray(bundle["bridge"], dtype=np.float32)
    bridge_shape = bridge_values.shape[:-1]
    log_ratios = predict_class_log_ratios(
        classifier, bridge_values.reshape(-1, bridge_values.shape[-1])
    ).reshape(*bridge_shape, 2)
    raw_evidence = (
        np.asarray(bundle["bridge_base"], dtype=np.float64)
        + log_ratios[..., 1] - log_ratios[..., 0]
    )
    if raw_evidence.shape[1] < 2:
        raise ValueError("Deployed bridge diagnostic needs at least two IID candidates.")
    result = {
        "norm_cross": float(cross),
        "norm_monitor": float(monitor),
        "bridge": float(np.var(raw_evidence, axis=1, ddof=1).mean()),
    }
    if not np.isfinite(list(result.values())).all():
        raise FloatingPointError("Non-finite deployed-classifier post-training check")
    return result


def pure_jana_bridge_diagnostic(bundle):
    """Evidence-path variance for the direct q_phi/q_eta baseline."""
    raw_evidence = np.asarray(bundle["bridge_base"], dtype=np.float64)
    if raw_evidence.ndim != 2 or raw_evidence.shape[1] < 2:
        raise ValueError("Pure-JANA bridge diagnostic needs grouped IID candidates.")
    value = float(np.var(raw_evidence, axis=1, ddof=1).mean())
    if not np.isfinite(value):
        raise FloatingPointError("Non-finite pure-JANA bridge diagnostic")
    return value

def normalized_positive_weights(values):
    values = np.asarray(values, dtype=np.float64)
    total = values.sum(dtype=np.float64)
    if not np.isfinite(values).all() or not np.all(values > 0.0) or not np.isfinite(total) or total <= 0.0:
        raise FloatingPointError("Invalid positive importance weights")
    return values / total

def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=np.float64)
    return float(1.0 / np.square(weights).sum())

def observation_contexts(task, observation, widths, seed):
    discrete = str(task.name) in DISCRETE_TASKS
    n_jitter = int(campaign["observation_jitters"]) if discrete else 1
    contexts = np.repeat(
        np.asarray(observation, dtype=np.float32).reshape(1, -1), n_jitter, axis=0
    )
    if discrete:
        contexts = dequantize(contexts, widths, seed + 1)
    return require_finite_rows(
        str(task.name), "paired observation contexts", contexts
    ).astype(np.float32)

def draw_fold_posterior_pair(
    task, transform, observation, widths, q_phi, classifier,
    n_proposal, n_samples, seed,
):
    """Draw pure-JANA and softmax-odds-corrected posteriors from one q_phi bank."""
    x_context = observation_contexts(task, observation, widths, seed)
    n_jitter = len(x_context)
    draws_per_context = max(256, int(math.ceil(int(n_proposal) / n_jitter)))
    z_draws = draw_flow_mixture(
        q_phi, draws_per_context, x_context, seed + 2, allocation="iid"
    )
    z_flat = z_draws.reshape(-1, z_draws.shape[-1])
    x_flat = np.repeat(
        x_context[:, None, :], draws_per_context, axis=1
    ).reshape(-1, x_context.shape[-1])
    theta = transform.inverse(z_flat).astype(np.float32)

    jana_rng = np.random.default_rng(int(seed + 3))
    jana_index = jana_rng.choice(
        len(theta), size=int(n_samples), replace=int(n_samples) > len(theta)
    )

    points = np.column_stack([z_flat, x_flat])
    ratios = predict_class_probability_ratios(classifier, points)
    odds = ratios[:, 0].reshape(n_jitter, draws_per_context)
    all_weights = []
    for index in range(n_jitter):
        all_weights.append(normalized_positive_weights(odds[index]) / n_jitter)
    weights = normalized_positive_weights(np.concatenate(all_weights))
    hybrid_rng = np.random.default_rng(int(seed + 4))
    hybrid_index = hybrid_rng.choice(
        len(theta), size=int(n_samples), replace=True, p=weights
    )
    diagnostics = {
        "ESS": effective_sample_size(weights),
        "ESS_fraction": effective_sample_size(weights) / len(weights),
        "max_weight": float(weights.max()),
        "posthoc_rms_log_ZP": float(np.sqrt(np.mean(np.square(
            np.log(odds.mean(axis=1))
        )))),
        "jitter_context_weighting": "uniform_no_bridge",
        "observation_jitters": n_jitter,
        "shared_q_phi_proposal_rows": len(theta),
    }
    return {
        METHOD_JANA: theta[jana_index].astype(np.float32),
        METHOD_HYBRID: theta[hybrid_index].astype(np.float32),
    }, diagnostics

def draw_fold_hybrid_predictive(
    transform, posterior_theta, q_eta, classifier,
    n_outputs, n_candidates, seed,
):
    """Finite-K SIR using direct softmax odds D_S/D_L on q_eta candidates."""
    posterior_theta = np.asarray(posterior_theta, dtype=np.float32)
    n_outputs = int(n_outputs)
    n_candidates = int(n_candidates)
    if n_outputs < 1 or n_candidates < 2:
        raise ValueError("Predictive SIR needs positive outputs and at least two candidates.")
    rng = np.random.default_rng(int(seed))
    chosen_theta = posterior_theta[
        rng.choice(len(posterior_theta), size=n_outputs, replace=n_outputs > len(posterior_theta))
    ]
    z = transform.forward(chosen_theta)
    x_candidates = draw_flow_mixture(
        q_eta, n_candidates, z, seed + 1, allocation="iid"
    )
    z_repeat = np.repeat(z[:, None, :], n_candidates, axis=1)
    points = np.concatenate([z_repeat, x_candidates], axis=2).reshape(
        -1, z.shape[1] + x_candidates.shape[2]
    )
    ratios = predict_class_probability_ratios(classifier, points)
    odds = ratios[:, 1].reshape(n_outputs, n_candidates)
    row_sums = odds.sum(axis=1, keepdims=True)
    if not np.isfinite(row_sums).all() or not np.all(row_sums > 0.0):
        raise FloatingPointError("Invalid predictive softmax-odds normalization")
    weights = odds / row_sums
    cdf = np.cumsum(weights, axis=1)
    uniforms = rng.random(n_outputs)
    selected = np.minimum(
        np.sum(cdf < uniforms[:, None], axis=1), n_candidates - 1
    )
    x_selected = x_candidates[np.arange(n_outputs), selected]
    candidate_ess = 1.0 / np.square(weights).sum(axis=1)
    diagnostics = {
        "method": METHOD_HYBRID,
        "candidate_ESS_mean": float(candidate_ess.mean()),
        "candidate_ESS_fraction_mean": float((candidate_ess / n_candidates).mean()),
        "candidate_ESS_fraction_min_over_theta": float((candidate_ess / n_candidates).min()),
        "candidate_max_weight_mean": float(weights.max(axis=1).mean()),
        "candidate_max_weight_worst": float(weights.max()),
        "posthoc_candidate_log_ZL_rms": float(np.sqrt(np.mean(np.square(
            np.log(odds.mean(axis=1))
        )))),
        "candidate_count": n_candidates,
        "outputs": n_outputs,
    }
    return chosen_theta.astype(np.float32), x_selected.astype(np.float32), diagnostics

def draw_fold_jana_predictive(
    transform, posterior_theta, q_eta, n_outputs, seed,
):
    """Direct pure-JANA composition q_phi(theta|x) then q_eta(x|theta)."""
    posterior_theta = np.asarray(posterior_theta, dtype=np.float32)
    n_outputs = int(n_outputs)
    if n_outputs < 1:
        raise ValueError("Pure-JANA predictive sampling needs positive outputs.")
    rng = np.random.default_rng(int(seed))
    chosen_theta = posterior_theta[
        rng.choice(len(posterior_theta), size=n_outputs, replace=n_outputs > len(posterior_theta))
    ]
    z = transform.forward(chosen_theta)
    x_selected = draw_flow_mixture(
        q_eta, 1, z, seed + 1, allocation="iid"
    )[:, 0, :]
    diagnostics = {
        "method": METHOD_JANA,
        "candidate_count": 1,
        "outputs": n_outputs,
        "sampling": "direct_normalized_q_eta",
    }
    return chosen_theta.astype(np.float32), x_selected.astype(np.float32), diagnostics

def _reference_standardization(reference):
    reference = np.asarray(reference, dtype=np.float64)
    mean = reference.mean(axis=0)
    std = reference.std(axis=0)
    std = np.where(std > 1.0e-8, std, 1.0)
    return mean, std

def _reference_only_bandwidth2(reference_standardized):
    pilot = np.asarray(reference_standardized, dtype=np.float64)
    if len(pilot) > 1_000:
        pilot = pilot[:1_000]
    distances = pdist(pilot, metric="sqeuclidean")
    distances = distances[distances > 0]
    return max(float(np.median(distances)) if len(distances) else 1.0, 1.0e-8)

def _mmd2_with_bandwidth(left, right, bandwidth2, chunk=256):
    def kernel_mean(first, second):
        total = 0.0
        count = 0
        for start in range(0, len(first), int(chunk)):
            d2 = cdist(first[start:start + int(chunk)], second, metric="sqeuclidean")
            total += float(np.exp(-d2 / (2.0 * bandwidth2)).sum())
            count += d2.size
        return total / count

    value = (
        kernel_mean(left, left) + kernel_mean(right, right)
        - 2.0 * kernel_mean(left, right)
    )
    return max(0.0, float(value))

def paired_distribution_metrics(reference, candidates, seed, max_samples=2_000):
    """Evaluate both methods using one reference subset and one RBF kernel."""
    from sbibm.metrics import c2st

    if tuple(candidates) != METHODS:
        raise ValueError(f"Expected ordered candidates {METHODS}, got {tuple(candidates)}")
    reference = np.asarray(reference, dtype=np.float64)
    candidate_arrays = {
        method: np.asarray(candidates[method], dtype=np.float64) for method in METHODS
    }
    n = min(
        len(reference), int(max_samples),
        *(len(candidate_arrays[method]) for method in METHODS),
    )
    if n < 32:
        raise RuntimeError("Too few paired rows for stable MMD/C2ST evaluation.")
    rng = np.random.default_rng(int(seed))
    reference_equal = reference[rng.choice(len(reference), n, replace=False)]
    candidate_equal = {}
    for index, method in enumerate(METHODS):
        method_rng = np.random.default_rng(int(seed + 1_009 * (index + 1)))
        values = candidate_arrays[method]
        candidate_equal[method] = values[
            method_rng.choice(len(values), n, replace=False)
        ]

    mean, std = _reference_standardization(reference)
    reference_standardized = (reference_equal - mean) / std
    bandwidth2 = _reference_only_bandwidth2(reference_standardized)
    results = {}
    for method in METHODS:
        candidate_standardized = (candidate_equal[method] - mean) / std
        mmd2 = _mmd2_with_bandwidth(
            reference_standardized, candidate_standardized, bandwidth2
        )
        score = float(c2st(
            torch.as_tensor(reference_equal, dtype=torch.float32),
            torch.as_tensor(candidate_equal[method], dtype=torch.float32),
            seed=int(seed + 20_000), n_folds=5, z_score=True,
        ).item())
        results[method] = {
            "C2ST": score,
            "MMD2": mmd2,
            "MMD": math.sqrt(max(0.0, mmd2)),
            "MMD_bandwidth2": bandwidth2,
            "MMD_n": n,
            "MMD_bandwidth_source": "reference_only_shared",
        }
    return results

In [ ]:
METHOD_COLORS = {METHOD_JANA: "C0", METHOD_HYBRID: "C1"}
METHOD_MARKERS = {METHOD_JANA: "o", METHOD_HYBRID: "D"}

def export_figure(fig, output_dir, script_name):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = export_standalone_figure_script(fig, script_name=script_name, output_dir=output_dir)
    fig.savefig(output_dir / f"{Path(script_name).stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(output_dir / f"{Path(script_name).stem}.pdf", bbox_inches="tight")
    print("Exported:", path)
    return path

def plot_posterior(reference, candidates, labels, output_dir, observation_number):
    reference = np.asarray(reference)
    candidates = {method: np.asarray(candidates[method]) for method in METHODS}
    rng = np.random.default_rng(SEED + observation_number)
    if reference.shape[1] == 2:
        fig, axes = plt.subplots(1, 4, figsize=(15.4, 3.7), constrained_layout=True)
        panels = [
            (axes[0], reference, "official reference", "black"),
            (axes[1], candidates[METHOD_JANA], METHOD_LABELS[METHOD_JANA], METHOD_COLORS[METHOD_JANA]),
            (axes[2], candidates[METHOD_HYBRID], METHOD_LABELS[METHOD_HYBRID], METHOD_COLORS[METHOD_HYBRID]),
        ]
        for ax, values, title, color in panels:
            index = rng.choice(len(values), min(3_000, len(values)), replace=False)
            ax.scatter(
                values[index, 0], values[index, 1], s=4, alpha=0.22,
                color=color, rasterized=True,
            )
            ax.set(xlabel=labels[0], ylabel=labels[1], title=title)
        axes[3].hist(
            reference[:, 0], bins=60, density=True, histtype="step",
            lw=2, color="black", label="reference",
        )
        for method in METHODS:
            axes[3].hist(
                candidates[method][:, 0], bins=60, density=True,
                histtype="step", lw=2, color=METHOD_COLORS[method],
                label=METHOD_LABELS[method],
            )
        axes[3].set(xlabel=labels[0], ylabel="density", title="first marginal")
        axes[3].legend(fontsize=8)
    else:
        n_show = min(4, reference.shape[1])
        fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0), constrained_layout=True)
        for index, ax in enumerate(axes.flat):
            if index >= n_show:
                ax.axis("off")
                continue
            ax.hist(
                reference[:, index], bins=50, density=True, histtype="step",
                lw=2, color="black", label="reference",
            )
            for method in METHODS:
                ax.hist(
                    candidates[method][:, index], bins=50, density=True,
                    histtype="step", lw=2, color=METHOD_COLORS[method],
                    label=METHOD_LABELS[method],
                )
            ax.set(xlabel=labels[index], ylabel="density", title=f"marginal {index + 1}")
            ax.legend(fontsize=8)
    export_figure(fig, output_dir, f"posterior_observation_{observation_number}")
    plt.show()

def plot_predictive(
    reference_theta, reference_x, candidates,
    output_dir, observation_number,
):
    reference_theta = np.asarray(reference_theta)
    reference_x = np.asarray(reference_x)
    candidates = {
        method: (np.asarray(candidates[method][0]), np.asarray(candidates[method][1]))
        for method in METHODS
    }
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), constrained_layout=True)
    n_marginals = min(3, reference_x.shape[1])
    for index in range(3):
        ax = axes.flat[index]
        if index >= n_marginals:
            ax.axis("off")
            continue
        ax.hist(
            reference_x[:, index], bins=50, density=True, histtype="step",
            lw=2, color="black", label="official predictive",
        )
        for method in METHODS:
            ax.hist(
                candidates[method][1][:, index], bins=50, density=True,
                histtype="step", lw=2, color=METHOD_COLORS[method],
                label=METHOD_LABELS[method],
            )
        ax.set(xlabel=fr"$x_{{{index + 1}}}$", ylabel="density", title=f"predictive marginal {index + 1}")
        ax.legend(fontsize=8)
    ax = axes.flat[3]
    rng = np.random.default_rng(SEED + 90_000 + int(observation_number))
    joint_panels = [(reference_theta, reference_x, "black", "official joint")]
    joint_panels.extend([
        (candidates[method][0], candidates[method][1], METHOD_COLORS[method], METHOD_LABELS[method])
        for method in METHODS
    ])
    for theta, values, color, label in joint_panels:
        use = rng.choice(len(values), min(2_000, len(values)), replace=False)
        ax.scatter(
            theta[use, 0], values[use, 0], s=6, alpha=0.20,
            color=color, label=label, rasterized=True,
        )
    ax.set(xlabel=r"$\theta_1$", ylabel=r"$x_1$", title="posterior-predictive joint slice")
    ax.legend(fontsize=8)
    export_figure(fig, output_dir, f"predictive_observation_{observation_number}")
    plt.show()

def plot_training_histories(histories, diagnostic_rows, output_dir):
    fig, axes = plt.subplots(1, 4, figsize=(17.0, 3.8), constrained_layout=True)
    for label, history in histories:
        axes[0].plot(history.get("validation_ce", []), alpha=0.55, label=label)
        axes[1].step(
            np.arange(1, len(history.get("learning_rate", [])) + 1),
            history.get("learning_rate", []), where="post", alpha=0.55,
        )
    constraint_frame = pd.DataFrame(diagnostic_rows)
    if not constraint_frame.empty:
        campaigns = constraint_frame["fold"].to_numpy() + 1
        axes[2].plot(
            campaigns, np.maximum(constraint_frame["audit_norm_monitor"], 1.0e-12),
            "o-", label="hybrid classifier",
        )
        axes[3].plot(
            campaigns, np.maximum(constraint_frame["jana_audit_bridge"], 1.0e-12),
            "o-", color=METHOD_COLORS[METHOD_JANA], label=METHOD_LABELS[METHOD_JANA],
        )
        axes[3].plot(
            campaigns, np.maximum(constraint_frame["audit_bridge"], 1.0e-12),
            "D-", color=METHOD_COLORS[METHOD_HYBRID], label=METHOD_LABELS[METHOD_HYBRID],
        )
    axes[0].set(title="held-out hybrid multiclass CE", xlabel="epoch", ylabel="CE objective")
    axes[1].set(title="progressive learning rate", xlabel="epoch", ylabel="Adam learning rate", yscale="log")
    axes[2].set(title="conditional-mass check", xlabel="matched campaign", ylabel="pooled squared error", yscale="log")
    axes[3].set(title="raw evidence-path consistency", xlabel="matched campaign", ylabel="variance", yscale="log")
    for ax in axes:
        ax.grid(alpha=0.25)
        if ax.lines and ax is not axes[1]:
            ax.legend(fontsize=7)
    export_figure(fig, output_dir, "training_and_post_training_checks")
    plt.show()
    return constraint_frame




## 4. Run the task selected by the public launcher notebook

As in Exercise 9a, the complete fixed 10,000-pair bank trains one four-member joint $q_\phi(z\mid x)$ mixture and one four-member joint $q_\eta(x\mid z)$ mixture. After those flows are frozen, complete matched $S/P/L$ groups are split 80/20 for classifier training and held-out pure-CE checkpoint selection. Both methods reuse these exact flow checkpoints. Pure JANA samples the normalized flows directly; the hybrid route alone evaluates the ten plain three-class MLPs and applies their arithmetically averaged finite-proposal softmax-odds correction.

The paired posterior helper constructs one jittered observation bank and one $q_\phi$ proposal bank. Direct-JANA rows are an unweighted subset; hybrid rows are sampled with $D_S/D_P$ weights. For posterior prediction, pure JANA composes a direct $q_\phi$ draw with one direct $q_\eta$ draw, while the hybrid method uses finite-$K$ $D_S/D_L$ SIR. No explicit exponential of a classifier logit or log ratio is used for either finite-bank correction.

The official reference posterior remains hidden until both methods and all checkpoints are frozen. A single official posterior-predictive simulation bank is then shared by both methods, so adding JANA does not double the evaluation simulator budget. Every task writes long-form rows keyed by `method`, paired sample files, diagnostics, figures, and standalone plotting scripts to the explicit persistent artifact root printed above.


In [ ]:
def train_and_evaluate_task(task_name):
    task_index = ALL_TASKS.index(task_name)
    task_seed = SEED + 100_000 * task_index
    task = sbibm.get_task(task_name)
    backend_suffix = (
        "" if SIMULATOR_BACKEND == "sbibm_default"
        else f"__{SIMULATOR_BACKEND}"
    )
    cache_path = CACHE_ROOT / (
        f"{task_name}_prior_predictive_{campaign['num_simulations']}"
        f"_seed{task_seed}{backend_suffix}.npz"
    )
    print(f"\n{task_name}: exact persistent cache path is {cache_path}")
    try:
        bank = load_or_simulate_bank(
            task, campaign["num_simulations"], cache_path=cache_path,
            seed=task_seed, chunk_size=20_000,
        )
    except Exception as exc:
        if task_name in {"sir", "lotka_volterra"}:
            raise RuntimeError(
                f"Configured {task_name} training-bank stage unavailable. Check the "
                f"task-specific simulator preflight or place the exact cache at {cache_path}; "
                "a working simulator is still required later for predictive evaluation."
            ) from exc
        raise

    require_finite_rows(task_name, "training bank theta", bank.theta)
    require_finite_rows(task_name, "training bank x", bank.x)
    transform = infer_parameter_transform(task)
    z_all = require_finite_rows(
        task_name, "transformed training parameters", transform.forward(bank.theta)
    ).astype(np.float32)
    widths = dequantization_widths(task_name, bank.x)
    x_all = require_finite_rows(
        task_name, "dequantized training observations",
        dequantize(bank.x, widths, task_seed + 1),
    ).astype(np.float32)
    observations = {}
    for observation_number in campaign["observations"]:
        observation = task.get_observation(num_observation=int(observation_number))
        observation_array = observation.detach().cpu().numpy().reshape(1, -1).astype(np.float32)
        observations[int(observation_number)] = require_finite_rows(
            task_name, f"official observation {observation_number}", observation_array
        ).astype(np.float32)

    fold_draws = {
        method: {number: [] for number in observations} for method in METHODS
    }
    fold_hybrid_diagnostics = {number: [] for number in observations}
    fold_predictive_pairs = {
        method: {number: [] for number in observations} for method in METHODS
    }
    fold_predictive_diagnostics = []
    histories = []
    diagnostic_rows = []
    for fold in range(campaign["n_folds"]):
        print("\n" + "=" * 90)
        print(f"{task_name}: matched campaign {fold + 1}/{campaign['n_folds']}")
        if int(campaign["n_folds"]) == 1:
            # Match Exercise 9a: one common 10k simulator bank trains the frozen
            # four-member q_phi/q_eta mixtures. Class groups are then split by
            # complete S/P/L group for CE selection and a post-training audit.
            flow_index = np.arange(len(bank.theta), dtype=np.int64)
            ratio_index = np.arange(len(bank.theta), dtype=np.int64)
        else:
            split = split_simulation_bank(
                bank, seed=task_seed + 17, fold=fold,
                n_folds=campaign["n_folds"],
            )
            flow_index = split["flow_indices"]
            ratio_index = split["ratio_indices"]
        rng = np.random.default_rng(task_seed + 100 + fold)
        order = rng.permutation(ratio_index)
        n_validation = max(24, int(round(0.20 * len(order))))
        if n_validation >= len(order) - 24:
            raise RuntimeError("Too few ratio rows for the grouped 80/20 split")
        validation_index = order[:n_validation]
        training_index = order[n_validation:]
        # As in 9a, consistency banks are constructed only after model
        # selection from held-out group rows. They never enter gradients.
        audit_index = validation_index
        split_parts = [training_index, validation_index]
        if (
            int(campaign["n_folds"]) > 1
            and np.intersect1d(flow_index, ratio_index).size
        ) or any(
            np.intersect1d(split_parts[i], split_parts[j]).size
            for i in range(len(split_parts)) for j in range(i + 1, len(split_parts))
        ):
            raise RuntimeError("Flow/ratio or classifier group-split leakage")

        fold_dir = MODEL_ROOT / task_name / RUN_TAG / f"fold_{fold:02d}"
        q_phi = train_spline_flow_ensemble(
            z_all[flow_index], context=x_all[flow_index],
            checkpoint=fold_dir / "q_phi.pt", ensemble_size=campaign["flow_members"],
            model_config=flow_model_config(z_all.shape[1]),
            training_config=FLOW_TRAINING_CONFIG, device=device,
            seed=task_seed + 1_000 + 100 * fold,
            load_if_available=LOAD_IF_AVAILABLE, verify_checkpoint_data=True,
        )
        q_eta = train_spline_flow_ensemble(
            x_all[flow_index], context=z_all[flow_index],
            checkpoint=fold_dir / "q_eta.pt", ensemble_size=campaign["flow_members"],
            model_config=flow_model_config(x_all.shape[1]),
            training_config=FLOW_TRAINING_CONFIG, device=device,
            seed=task_seed + 2_000 + 100 * fold,
            load_if_available=LOAD_IF_AVAILABLE, verify_checkpoint_data=True,
        )
        train_groups = build_class_groups(
            z_all[training_index], x_all[training_index], q_phi, q_eta,
            task_seed + 3_000 + 100 * fold,
        )
        validation_groups = build_class_groups(
            z_all[validation_index], x_all[validation_index], q_phi, q_eta,
            task_seed + 4_000 + 100 * fold,
        )
        classifier = train_plain_classifier(
            train_groups, validation_groups,
            fold_dir / "plain_three_class_classifier", task_seed + 7_000 + 100 * fold,
        )
        for member, pack in enumerate(classifier):
            histories.append((f"fold {fold + 1}, member {member + 1}", pack["history"]))

        # Construct all normalization/bridge banks only after hybrid CE model
        # selection. They cannot enter gradients or checkpoint eligibility.
        audit_bundle = build_post_training_diagnostic_bundle(
            task, transform, z_all[audit_index], x_all[audit_index], q_phi, q_eta,
            task_seed + 6_000 + 100 * fold,
        )
        prepared_audit = prepare_diagnostic_bundle(
            audit_bundle, classifier[0]["center"], classifier[0]["scale"]
        )
        member_class_validation = [
            validation_classification_objective(
                pack["model"],
                torch.as_tensor(
                    transform_classifier_points(
                        validation_groups, pack["center"], pack["scale"]
                    ), dtype=torch.float32,
                ),
            )
            for pack in classifier
        ]
        member_consistency = [
            member_constraint_diagnostics(pack["model"], prepared_audit)
            for pack in classifier
        ]
        deployed_audit = deployed_classifier_constraint_diagnostics(
            classifier, audit_bundle
        )
        jana_audit_bridge = pure_jana_bridge_diagnostic(audit_bundle)
        diagnostic_rows.append({
            "task": task_name, "fold": fold,
            "audit_rows": len(audit_index),
            "audit_norm_cross": deployed_audit["norm_cross"],
            "audit_norm_monitor": deployed_audit["norm_monitor"],
            "audit_bridge": deployed_audit["bridge"],
            "jana_audit_bridge": jana_audit_bridge,
            "member_mean_audit_norm_cross": float(np.mean([row["norm_cross"] for row in member_consistency])),
            "member_mean_audit_norm_monitor": float(np.mean([row["norm_monitor"] for row in member_consistency])),
            "member_mean_audit_bridge": float(np.mean([row["bridge"] for row in member_consistency])),
            "member_mean_validation_CE": float(np.mean(member_class_validation)),
            "diagnostics_constructed_after_checkpoint": True,
        })

        for observation_number, observation in observations.items():
            paired_draws, hybrid_diagnostics = draw_fold_posterior_pair(
                task, transform, observation, widths, q_phi, classifier,
                campaign["n_proposal"], campaign["n_posterior"],
                task_seed + 10_000 + 1_000 * fold + observation_number,
            )
            for method in METHODS:
                fold_draws[method][observation_number].append(paired_draws[method])
            hybrid_diagnostics.update({
                "method": METHOD_HYBRID,
                "fold": fold, "num_observation": observation_number,
            })
            fold_hybrid_diagnostics[observation_number].append(hybrid_diagnostics)

            n_predictive_fold = int(math.ceil(
                campaign["predictive_samples"] / campaign["n_folds"]
            ))
            jana_theta, jana_x, jana_diagnostics = draw_fold_jana_predictive(
                transform, paired_draws[METHOD_JANA], q_eta, n_predictive_fold,
                task_seed + 14_000 + 1_000 * fold + observation_number,
            )
            hybrid_theta, hybrid_x, hybrid_predictive_diagnostics = draw_fold_hybrid_predictive(
                transform, paired_draws[METHOD_HYBRID], q_eta, classifier,
                n_predictive_fold, campaign["predictive_candidates"],
                task_seed + 15_000 + 1_000 * fold + observation_number,
            )
            fold_predictive_pairs[METHOD_JANA][observation_number].append(
                (jana_theta, jana_x)
            )
            fold_predictive_pairs[METHOD_HYBRID][observation_number].append(
                (hybrid_theta, hybrid_x)
            )
            for row in (jana_diagnostics, hybrid_predictive_diagnostics):
                row.update({"fold": fold, "num_observation": observation_number})
                fold_predictive_diagnostics.append(row)

        del q_phi, q_eta, classifier, train_groups, validation_groups
        del audit_bundle, prepared_audit
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    figure_dir = FIGURE_ROOT / task_name / RUN_TAG
    diagnostic_table = plot_training_histories(histories, diagnostic_rows, figure_dir)
    diagnostic_table.to_csv(
        RESULT_ROOT / f"{task_name}__{RUN_TAG}__post_training_checks.csv", index=False
    )
    deployed_check_summary = {
        "mean_hybrid_audit_norm_cross": float(diagnostic_table["audit_norm_cross"].mean()),
        "mean_hybrid_audit_norm_monitor": float(diagnostic_table["audit_norm_monitor"].mean()),
        "mean_hybrid_audit_bridge": float(diagnostic_table["audit_bridge"].mean()),
        "mean_jana_audit_bridge": float(diagnostic_table["jana_audit_bridge"].mean()),
    }
    predictive_diagnostic_table = pd.DataFrame(fold_predictive_diagnostics)
    predictive_diagnostic_path = RESULT_ROOT / (
        f"{task_name}__{RUN_TAG}__predictive_diagnostics.csv"
    )
    predictive_diagnostic_table.to_csv(predictive_diagnostic_path, index=False)
    print("Saved paired predictive diagnostics:", predictive_diagnostic_path)

    rows = []
    labels = task.get_labels_parameters()
    for observation_number in observations:
        candidates = {}
        for method_index, method in enumerate(METHODS):
            pool = np.concatenate(fold_draws[method][observation_number])
            rng = np.random.default_rng(
                task_seed + 20_000 + 1_000 * method_index + observation_number
            )
            candidates[method] = pool[
                rng.choice(len(pool), campaign["n_posterior"], replace=False)
            ]
            candidates[method] = require_finite_rows(
                task_name, f"{method} posterior observation {observation_number}",
                candidates[method],
            ).astype(np.float32)

        reference = task.get_reference_posterior_samples(
            num_observation=int(observation_number)
        ).detach().cpu().numpy().astype(np.float32)
        reference = require_finite_rows(
            task_name, f"official posterior observation {observation_number}", reference
        ).astype(np.float32)
        posterior_results = paired_distribution_metrics(
            reference, candidates, task_seed + 30_000 + observation_number
        )
        posterior_path = RESULT_ROOT / (
            f"{task_name}__{RUN_TAG}__observation{observation_number}"
            "__posterior_samples.npz"
        )
        np.savez_compressed(
            posterior_path,
            jana_theta=candidates[METHOD_JANA],
            hybrid_theta=candidates[METHOD_HYBRID],
            reference_theta=reference,
            method_keys=np.asarray(METHODS),
            observation_number=np.asarray(observation_number),
            campaign_seed=np.asarray(SEED),
            campaign_signature=np.asarray(CAMPAIGN_SIGNATURE),
            simulator_backend=np.asarray(SIMULATOR_BACKEND),
            metric_schema=np.asarray(METRIC_SCHEMA),
        )
        print("Saved paired posterior samples:", posterior_path)

        predictive_pools = {}
        for method in METHODS:
            predictive_pools[method] = (
                np.concatenate([
                    pair[0] for pair in fold_predictive_pairs[method][observation_number]
                ]),
                np.concatenate([
                    pair[1] for pair in fold_predictive_pairs[method][observation_number]
                ]),
            )
        n_predictive_eval = min(
            int(campaign["predictive_reference_calls"]), len(reference),
            *(len(predictive_pools[method][0]) for method in METHODS),
        )
        if n_predictive_eval < 32:
            raise RuntimeError("Too few posterior-predictive rows for a stable evaluation.")
        candidate_predictive = {}
        for method_index, method in enumerate(METHODS):
            predictive_rng = np.random.default_rng(
                task_seed + 40_000 + 1_000 * method_index + observation_number
            )
            theta_pool, x_pool = predictive_pools[method]
            candidate_index = predictive_rng.choice(
                len(theta_pool), n_predictive_eval, replace=False
            )
            candidate_predictive[method] = (
                require_finite_rows(
                    task_name, f"{method} predictive theta observation {observation_number}",
                    theta_pool[candidate_index],
                ).astype(np.float32),
                require_finite_rows(
                    task_name, f"{method} predictive x observation {observation_number}",
                    x_pool[candidate_index],
                ).astype(np.float32),
            )

        reference_rng = np.random.default_rng(
            task_seed + 42_000 + observation_number
        )
        reference_index = reference_rng.choice(
            len(reference), n_predictive_eval, replace=False
        )
        reference_predictive_theta = reference[reference_index]

        # One evaluation-only official simulator bank is shared by both methods.
        seed_everything(task_seed + 50_000 + observation_number)
        simulator = task.get_simulator(max_calls=n_predictive_eval)
        reference_predictive_raw = simulator(
            torch.as_tensor(reference_predictive_theta, dtype=torch.float32)
        )
        require_finite_rows(
            task_name,
            f"official predictive raw observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_raw,
        )
        reference_predictive_x = task.flatten_data(
            reference_predictive_raw
        ).detach().cpu().numpy().astype(np.float32)
        reference_predictive_x = require_finite_rows(
            task_name,
            f"official predictive flattened observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_x,
        ).astype(np.float32)
        reference_predictive_x = dequantize(
            reference_predictive_x, widths,
            task_seed + 60_000 + observation_number,
        )
        reference_predictive_x = require_finite_rows(
            task_name,
            f"official predictive evaluation observation {observation_number} ({n_predictive_eval} calls)",
            reference_predictive_x,
        ).astype(np.float32)
        print(
            f"{task_name} observation {observation_number}: "
            f"{n_predictive_eval:,} shared evaluation-only simulator calls "
            f"(outside the {campaign['num_simulations']:,}-call training bank)."
        )

        predictive_x_results = paired_distribution_metrics(
            reference_predictive_x,
            {method: candidate_predictive[method][1] for method in METHODS},
            task_seed + 70_000 + observation_number,
        )
        reference_joint = np.column_stack([
            reference_predictive_theta, reference_predictive_x
        ])
        candidate_joint = {
            method: np.column_stack(candidate_predictive[method]) for method in METHODS
        }
        predictive_joint_results = paired_distribution_metrics(
            reference_joint, candidate_joint,
            task_seed + 80_000 + observation_number,
        )
        hybrid_diagnostics = pd.DataFrame(
            fold_hybrid_diagnostics[observation_number]
        ).mean(numeric_only=True).to_dict()
        hybrid_predictive_frame = predictive_diagnostic_table.loc[
            (predictive_diagnostic_table["method"] == METHOD_HYBRID)
            & (predictive_diagnostic_table["num_observation"] == observation_number)
        ]
        hybrid_predictive_diagnostics = hybrid_predictive_frame.mean(
            numeric_only=True
        ).to_dict()

        predictive_path = RESULT_ROOT / (
            f"{task_name}__{RUN_TAG}__observation{observation_number}"
            "__predictive_samples.npz"
        )
        np.savez_compressed(
            predictive_path,
            jana_theta=candidate_predictive[METHOD_JANA][0],
            jana_x=candidate_predictive[METHOD_JANA][1],
            hybrid_theta=candidate_predictive[METHOD_HYBRID][0],
            hybrid_x=candidate_predictive[METHOD_HYBRID][1],
            reference_theta=reference_predictive_theta,
            reference_x=reference_predictive_x,
            method_keys=np.asarray(METHODS),
            dequantization_widths=widths,
            evaluation_simulator_calls=np.asarray(n_predictive_eval),
            hybrid_predictive_candidates_per_theta=np.asarray(
                campaign["predictive_candidates"]
            ),
            campaign_seed=np.asarray(SEED),
            campaign_signature=np.asarray(CAMPAIGN_SIGNATURE),
            simulator_backend=np.asarray(SIMULATOR_BACKEND),
            metric_schema=np.asarray(METRIC_SCHEMA),
        )
        print("Saved paired predictive samples:", predictive_path)
        plot_posterior(
            reference, candidates, labels, figure_dir, observation_number
        )
        plot_predictive(
            reference_predictive_theta, reference_predictive_x,
            candidate_predictive, figure_dir, observation_number,
        )

        for method in METHODS:
            is_hybrid = method == METHOD_HYBRID
            method_predictive_diagnostics = (
                hybrid_predictive_diagnostics if is_hybrid else {}
            )
            rows.append({
                "task": task_name, "profile": PROFILE, "run_tag": RUN_TAG,
                "seed": SEED, "campaign_schema": CAMPAIGN_SCHEMA,
                "campaign_signature": CAMPAIGN_SIGNATURE,
                "metric_schema": METRIC_SCHEMA,
                "jana_paper_commit": JANA_PAPER_COMMIT,
                "method": method, "method_label": METHOD_LABELS[method],
                "num_simulations": campaign["num_simulations"],
                "flow_members": campaign["flow_members"],
                "classifier_members": campaign["classifier_members"],
                "num_observation": observation_number,
                "simulator_backend": SIMULATOR_BACKEND,
                "data_semantics": "dequantized_continuous_density" if task_name in DISCRETE_TASKS else "continuous_density",
                "dequantization_exact_unit_cell": task_name in {"bernoulli_glm_raw", "sir"},
                "dequantization_width_min": float(widths.min()),
                "dequantization_width_max": float(widths.max()),
                "classifier_objective": (
                    "equal_prior_multiclass_CE_only" if is_hybrid
                    else "none_pure_JANA_flow_NLL"
                ),
                "checkpoint_objective": (
                    "held_out_multiclass_CE_only" if is_hybrid
                    else "held_out_flow_NLL"
                ),
                "normalization_role": (
                    "post_training_self_normalized_finite_proposals_and_check" if is_hybrid
                    else "normalized_flow_density_no_ratio_correction"
                ),
                "bridge_role": "held_out_consistency_check_only",
                "uses_ce_correction": is_hybrid,
                "posterior_method": (
                    "q_phi_softmax_S_over_P_finite_proposal" if is_hybrid
                    else "direct_q_phi"
                ),
                "predictive_method": (
                    "q_eta_softmax_S_over_L_finite_K_SIR" if is_hybrid
                    else "direct_q_eta"
                ),
                "predictive_candidate_allocation": "iid",
                "predictive_candidates_per_theta": (
                    campaign["predictive_candidates"] if is_hybrid else 1
                ),
                "predictive_evaluation_simulator_calls": n_predictive_eval,
                "predictive_reference_shared_between_methods": True,
                "predictive_simulator_calls_in_training": 0,
                "predictive_candidate_max_weight_worst": (
                    float(hybrid_predictive_frame["candidate_max_weight_worst"].max())
                    if is_hybrid else np.nan
                ),
                "predictive_candidate_ESS_fraction_min_over_theta": (
                    float(hybrid_predictive_frame["candidate_ESS_fraction_min_over_theta"].min())
                    if is_hybrid else np.nan
                ),
                "C2ST": posterior_results[method]["C2ST"],
                "MMD2": posterior_results[method]["MMD2"],
                "MMD": posterior_results[method]["MMD"],
                **{f"posterior_{key}": value for key, value in posterior_results[method].items()},
                **{f"predictive_x_{key}": value for key, value in predictive_x_results[method].items()},
                **{f"predictive_joint_{key}": value for key, value in predictive_joint_results[method].items()},
                **{f"mean_hybrid_{key}": value for key, value in hybrid_diagnostics.items() if key not in {"fold", "num_observation"}},
                **{f"mean_predictive_{key}": value for key, value in method_predictive_diagnostics.items() if key not in {"fold", "num_observation"}},
                **deployed_check_summary,
            })
    result = pd.DataFrame(rows)
    result_path = RESULT_ROOT / f"{task_name}__{RUN_TAG}__metrics.csv"
    result.to_csv(result_path, index=False)
    print("Saved paired long-form metrics:", result_path)
    display(result.style.format(precision=4).hide(axis="index"))
    return result

In [ ]:
campaign_results = []
campaign_failures = []
for selected_task in TASKS_TO_RUN:
    status_path = RESULT_ROOT / f"{selected_task}__{RUN_TAG}__status.json"
    status_base = {
        **expected_status(selected_task, PROFILE, SEED),
        "simulator_backend": SIMULATOR_BACKEND,
        "simulator_preflight": SIMULATOR_BACKEND_DIAGNOSTICS,
    }
    status_path.write_text(
        json.dumps({**status_base, "status": "running"}, indent=2),
        encoding="utf-8",
    )
    print("Wrote running status manifest:", status_path)
    try:
        result = train_and_evaluate_task(selected_task)
        campaign_results.append(result)
        status = {**status_base, "status": "completed"}
    except Exception as exc:
        status = {
            **status_base,
            "status": "failed",
            "exception_type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        }
        campaign_failures.append(status)
        print(status["traceback"])
    status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("Wrote status manifest:", status_path)

if campaign_failures:
    failure_table = pd.DataFrame(campaign_failures).drop(columns="traceback")
    display(failure_table)
    if FAIL_ON_TASK_ERROR:
        raise RuntimeError(
            "The requested sbibm task failed. See the displayed table and "
            "the task-specific status JSON; the failure was not skipped."
        )

## Task hand-off

This task notebook has written a configuration-validated status manifest,
paired JANA/hybrid metrics, samples, models, figures, and standalone plotting
scripts under the shared Drive artifact root. Run `Exercise_9b_SBIBM.ipynb`
at any time to discover this result and compare it with whichever other task
notebooks have already finished under the same profile, seed, and tag.